# 1. Librerías

In [1]:
!pip install datasets -q

In [2]:
!pip install spacy -q

In [3]:
!python -m spacy download en_core_web_lg -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# 2. Carga del Dataset

In [4]:
from datasets import load_dataset
import pandas as pd
import spacy
import re
from tqdm import tqdm
from tqdm.auto import tqdm as tqdm_auto
tqdm_auto.pandas()

# Cargar dataset completo (train: 1,360,000 filas | test: 240,000 filas)
ds = load_dataset("adilbekovich/Sentiment140Twitter")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.csv:   0%|          | 0.00/107M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/18.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1360000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/240000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1360000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 240000
    })
})


# 3. Exploración inicial (EDA)

In [5]:
# Vista rápida del split train
df_preview = ds["train"].to_pandas()
print(f"Shape train: {df_preview.shape}")
print(f"\nDistribución de labels:")
print(df_preview["label"].value_counts())
print(f"\nPrimeras 5 filas:")
df_preview.head()

Shape train: (1360000, 2)

Distribución de labels:
label
1    680129
0    679871
Name: count, dtype: int64

Primeras 5 filas:


,text,label
0,Closet organizer install complete. Now for th...,0
1,Mornin' All!! ....I need to wake up....this w...,1
2,@Lega_c ahhhhhhhhhh! he suxxxxxxx! he claimed ...,0
3,@endlessblush Haha. I guess all the good bits ...,0
4,family guy funny,1


# 4. Preprocesamiento

In [6]:
# Cargar modelo en inglés — disable parser y ner para acelerar procesamiento
nlp = spacy.load("en_core_web_lg", disable=["parser", "ner"])
print(f"Modelo cargado: {nlp.meta['name']} v{nlp.meta['version']}")

Modelo cargado: core_web_lg v3.8.0


In [7]:
def clean_raw_text(text):
    """Elimina URLs, menciones @usuario y hashtags #tag"""
    text = re.sub(r"http\S+|www\S+", "", text)   # URLs
    text = re.sub(r"@\w+", "", text)              # Menciones
    text = re.sub(r"#\w+", "", text)              # Hashtags
    text = re.sub(r"\s+", " ", text).strip()      # Espacios extra
    return text


def batch_normalize(texts, batch_size=4000):
    """Tokeniza, lematiza y filtra stopwords usando nlp.pipe en batch"""
    results = []
    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size), total=len(texts)):
        tokens = [
            token.lemma_
            for token in doc
            if not token.is_stop    # Eliminar stopwords
            and token.is_alpha      # Solo letras
            and len(token) > 1      # Eliminar letras sueltas
        ]
        results.append(" ".join(tokens))
    return results


def preprocess_df(df):
    """Pipeline completo: limpieza raw -> normalización spaCy -> columnas finales"""
    df = df.copy()
    # Paso 1: limpieza de ruido Twitter
    df["text_clean"] = df["text"].progress_apply(clean_raw_text)
    # Paso 2: normalización con spaCy en batch
    df["text_normalized"] = batch_normalize(df["text_clean"].tolist())
    # Paso 3: quedarse solo con columnas finales
    df = df[["text_normalized", "label"]]
    return df

print("Funciones definidas correctamente.")

Funciones definidas correctamente.


In [8]:
# Convertir ambos splits a pandas
df_train = ds["train"].to_pandas()
df_test  = ds["test"].to_pandas()

print(f"Train: {df_train.shape}")
print(f"Test:  {df_test.shape}")

Train: (1360000, 2)
Test:  (240000, 2)


In [9]:
# Procesar TRAIN
print("Procesando TRAIN (1,360,000 filas)...")
df_train = preprocess_df(df_train)

print(f"\nShape final train: {df_train.shape}")
df_train.head()

Procesando TRAIN (1,360,000 filas)...


  0%|          | 0/1360000 [00:00<?, ?it/s]

100%|██████████| 1360000/1360000 [32:01<00:00, 707.89it/s] 



Shape final train: (1360000, 2)


,text_normalized,label
0,Closet organizer install complete torture go m...,0
1,Mornin need wake,1
2,ahhhhhhhhhh suxxxxxxx claim,0
3,haha guess good bit trailer,0
4,family guy funny,1


In [10]:
# Procesar TEST (~2-3 min)
print("Procesando TEST (240,000 filas)...")
df_test = preprocess_df(df_test)

print(f"\nShape final test: {df_test.shape}")
df_test.head()

Procesando TEST (240,000 filas)...


  0%|          | 0/240000 [00:00<?, ?it/s]

100%|██████████| 240000/240000 [05:46<00:00, 692.19it/s] 



Shape final test: (240000, 2)


,text_normalized,label
0,good morning nearly afternoon,1
1,keep plant watering,1
2,countdown begin,1
3,hey,1
4,see family todayy happy,1


# 5. Guardar Parquets

In [11]:
df_train.to_parquet("sentiment_preprocessed_train.parquet", index=False)
df_test.to_parquet("sentiment_preprocessed_test.parquet",  index=False)

print("Archivos guardados:")
print(f"  sentiment_preprocessed_train.parquet  -> {len(df_train):,} filas")
print(f"  sentiment_preprocessed_test.parquet   -> {len(df_test):,} filas")

Archivos guardados:
  sentiment_preprocessed_train.parquet  -> 1,360,000 filas
  sentiment_preprocessed_test.parquet   -> 240,000 filas


# 6. Verificación final

In [12]:
pd.set_option("display.max_colwidth", None)

print("=== TRAIN ===")
print(pd.read_parquet("sentiment_preprocessed_train.parquet").head(10))

print("\n=== TEST ===")
print(pd.read_parquet("sentiment_preprocessed_test.parquet").head(10))

=== TRAIN ===
                                                                              text_normalized  \
0                               Closet organizer install complete torture go mound old clothe   
1                                                                            Mornin need wake   
2                                                                 ahhhhhhhhhh suxxxxxxx claim   
3                                                                 haha guess good bit trailer   
4                                                                            family guy funny   
5                                       okay play DJ get ready day go list stuff online today   
6  buzi week finish test ready University Law examination mass unfinished scenario nearly die   
7                                                                            shizzel want sim   
8                                                                           wholesale enquiry   
9               